# Post-Training Explorer: Full Training Pipeline
**SNIA DSN AI Stack Webinar Series — 2026**

This notebook runs the complete post-training pipeline:
1. **SFT** (Supervised Fine-Tuning)
2. **DPO** (Direct Preference Optimization)
3. **GRPO** (Group Relative Policy Optimization)
4. **Export** — Generates `precomputed_results.json` for the web app
5. **ONNX Conversion** — Converts models for browser-side inference

**First step:** Run **Setup**, then the **Model Configuration** cell to select your model size (360M or 1.7B) and see GPU recommendations and runtime estimates.

Each section can be run independently. If you hit a Colab timeout during GRPO, run SFT+DPO in one session and GRPO in a second session.

In [ ]:
# ============================================================
# Setup: Install dependencies and prepare scripts
# ============================================================

# Install all required packages
!pip install -q torch transformers datasets accelerate peft trl sentencepiece safetensors

# Clone from GitHub (or pull latest if already cloned)
import os
PROJECT_ROOT = "/content/FineTuningDemo"
if os.path.exists(PROJECT_ROOT):
    print("Project already cloned — pulling latest changes...")
    os.chdir(PROJECT_ROOT)
    !git pull
else:
    !git clone https://github.com/provandal/post-training-explorer.git {PROJECT_ROOT}
    os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")
print(f"Scripts present: {os.path.exists('scripts/train_sft.py')}")

# List available scripts
if os.path.exists('scripts'):
    for f in sorted(os.listdir('scripts')):
        print(f"  scripts/{f}")
else:
    print("WARNING: scripts/ directory not found!")

In [ ]:
# ============================================================
# Model Configuration
# ============================================================
# Choose which SmolLM2 model to train. This selection flows
# through all subsequent cells automatically.
#
#   360M  — Fast training, fits on T4 GPU (~15 GB VRAM)
#           Good for learning the pipeline and quick experiments
#
#   1.7B  — Better accuracy, needs more VRAM
#           SFT/DPO fit on T4, GRPO benefits from A100
# ============================================================

MODEL_SIZE = "360M"  # <-- Change to "1.7B" for the larger model

# --- Derived settings (do not edit below this line) ---
MODEL_SIZE_FLAG = f"--model-size {MODEL_SIZE}"

_SLUGS = {"360M": "smollm2-360m", "1.7B": "smollm2-1.7b"}
if MODEL_SIZE == "360M":
    OUTPUT_DIR = "scripts/outputs"
else:
    OUTPUT_DIR = f"scripts/outputs/{_SLUGS[MODEL_SIZE]}"

# Runtime estimates
_RUNTIMES = {
    "360M": {
        "T4":   {"SFT": "~12 min", "DPO": "~8 min",  "GRPO": "~35 min",  "Export": "~3 min", "ONNX": "~5 min"},
        "A100": {"SFT": "~4 min",  "DPO": "~3 min",  "GRPO": "~10 min",  "Export": "~2 min", "ONNX": "~3 min"},
    },
    "1.7B": {
        "T4":   {"SFT": "~55 min", "DPO": "~35 min", "GRPO": "~150 min", "Export": "~8 min", "ONNX": "~10 min"},
        "A100": {"SFT": "~15 min", "DPO": "~10 min", "GRPO": "~45 min",  "Export": "~3 min", "ONNX": "~5 min"},
    },
}

print(f"Selected model: SmolLM2-{MODEL_SIZE}")
print(f"Output directory: {OUTPUT_DIR}")
print()

if MODEL_SIZE == "1.7B":
    print("NOTE: 1.7B model selected.")
    print("  SFT and DPO will work on T4, but A100 is recommended for GRPO.")
    print("  Go to Runtime > Change runtime type > A100 GPU")
    print()

print("Estimated runtimes:")
print(f"  {'Step':<8s} {'T4 GPU':<14s} {'A100 GPU':<14s}")
print(f"  {'─' * 36}")
for step in ["SFT", "DPO", "GRPO", "Export", "ONNX"]:
    t4 = _RUNTIMES[MODEL_SIZE]["T4"][step]
    a100 = _RUNTIMES[MODEL_SIZE]["A100"][step]
    print(f"  {step:<8s} {t4:<14s} {a100:<14s}")

In [ ]:
# ============================================================
# Verify GPU availability
# ============================================================

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → T4 GPU")
    print("Training will be extremely slow on CPU.")

In [ ]:
# ============================================================
# (Optional) HuggingFace Hub Login
# ============================================================
# Only needed for Step 5 (ONNX conversion + push to Hub).
# Skip this cell if you're only running SFT/DPO/GRPO/Export.
#
# Option A: Run this cell and paste your token when prompted.
# Option B: Set HF_TOKEN in Colab Secrets (key icon in sidebar).
# ============================================================

from huggingface_hub import login
login()

---
## Step 1: Supervised Fine-Tuning (SFT)

**What's the problem?** The base SmolLM2 model is a general-purpose language model — it has no idea what storage I/O workloads are. SFT is like **onboarding a new hire**: show them 480 labeled examples until they learn the pattern.

**Analogy:** You wouldn't expect a new storage engineer to classify workloads on day one. But after reviewing hundreds of examples with an experienced mentor, they learn the signatures — high random IOPS with small blocks means OLTP, large sequential writes mean backup, etc.

**What happens:**
- Generates 480 synthetic training examples (80 per workload class)
- Trains LoRA adapters (only ~0.5–0.9% of parameters) on attention layers
- Captures before/after model outputs for comparison

**What to expect:**
- Loss should drop from ~2.8 to below 0.5
- Sample accuracy depends on model size — 50–70% for 360M, higher for 1.7B
- The model will learn obvious distinctions (OLTP vs Backup) but may struggle with similar profiles (OLAP vs AI ML Training)

**What to look for in the results:**
- Are any classifications correct? When wrong, is the mistake reasonable?
- OLAP/AI ML confusion is expected — they have genuinely similar I/O signatures

**Expected runtime:** See the **Model Configuration** cell above

In [ ]:
# ============================================================
# Run SFT training
# ============================================================
# Trains the base SmolLM2 model to classify storage I/O
# workloads using LoRA adapters.
#
# Outputs saved to scripts/outputs/ (path varies by model size)
# ============================================================

!python scripts/train_sft.py {MODEL_SIZE_FLAG}

In [ ]:
# ============================================================
# Post-SFT Validation: Did the model learn the task?
# ============================================================

import json
import os
import matplotlib.pyplot as plt
import numpy as np

# --- Load artifacts ---
sft_dir = f'{OUTPUT_DIR}/sft'

loss_data = None
loss_path = os.path.join(sft_dir, 'training_loss.json')
if os.path.exists(loss_path):
    with open(loss_path) as f:
        loss_data = json.load(f)

comparison = None
comp_path = os.path.join(sft_dir, 'before_after_comparison.json')
if os.path.exists(comp_path):
    with open(comp_path) as f:
        comparison = json.load(f)

sanity = None
sanity_path = os.path.join(sft_dir, 'sft_sanity_check.json')
if os.path.exists(sanity_path):
    with open(sanity_path) as f:
        sanity = json.load(f)

# --- Two-panel figure ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Training loss curve
if loss_data and loss_data.get('loss_curve'):
    steps = [p['step'] for p in loss_data['loss_curve']]
    losses = [p['loss'] for p in loss_data['loss_curve']]
    ax1.plot(steps, losses, 'b-', linewidth=2)
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('SFT Training Loss')
    ax1.grid(True, alpha=0.3)
    if len(losses) >= 2:
        ax1.annotate(f'Start: {losses[0]:.2f}', xy=(steps[0], losses[0]),
                     fontsize=10, color='red', fontweight='bold')
        ax1.annotate(f'End: {losses[-1]:.2f}', xy=(steps[-1], losses[-1]),
                     fontsize=10, color='green', fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'No loss data available', ha='center', va='center', transform=ax1.transAxes)

# Right: Base vs SFT accuracy on sample prompts
if sanity and sanity.get('sample_results'):
    labels = [r['expected'][:12] for r in sanity['sample_results']]
    sft_correct = [1 if r['correct'] else 0 for r in sanity['sample_results']]
    # Base model is essentially random for this task
    base_correct = [0] * len(labels)  # Base model gets ~0% on structured task

    x = np.arange(len(labels))
    width = 0.35
    ax2.bar(x - width/2, base_correct, width, label='Base', color='#ff9999', alpha=0.8)
    ax2.bar(x + width/2, sft_correct, width, label='SFT', color='#66b3ff', alpha=0.8)
    ax2.set_ylabel('Correct (1) / Wrong (0)')
    ax2.set_title('Base vs SFT: Sample Accuracy')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax2.legend()
    ax2.set_ylim(-0.1, 1.3)
else:
    ax2.text(0.5, 0.5, 'No sanity check data', ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
plt.show()

# --- Sample outputs with badges ---
if comparison:
    print("\nSample SFT Outputs:")
    print("-" * 60)
    for c in comparison[:3]:
        correct = c.get('sft_correct', False)
        badge = "CORRECT" if correct else "WRONG"
        print(f"  True label:  {c['true_label']}")
        print(f"  SFT output:  {c['sft_output'][:120]}")
        print(f"  Result:      [{badge}]")
        print()

# --- Interpretation ---
if sanity:
    verdict = sanity.get('verdict', 'UNKNOWN')
    accuracy = sanity.get('sample_accuracy', 0)
    initial = sanity.get('initial_loss')
    final = sanity.get('final_loss')

    print("=" * 60)
    print("INTERPRETATION")
    print("=" * 60)
    if initial and final:
        reduction = (initial - final) / initial * 100
        print(f"  Loss dropped {reduction:.0f}% ({initial:.2f} -> {final:.2f})")
        print(f"  The model learned the classification task.")
    print(f"  Sample accuracy: {accuracy:.0%}")
    if accuracy >= 0.6:
        print(f"  Strong performance for SmolLM2-{MODEL_SIZE}.")
    elif accuracy >= 0.4:
        print(f"  Moderate performance — typical for this model size on 6-class task.")
    else:
        print(f"  Low accuracy — but SFT is just the foundation. GRPO will optimize.")
    print(f"\n  Verdict: {verdict}")
    print("=" * 60)

In [ ]:
# ============================================================
# SFT Probe History: Watch the Model Learn (Option C)
# ============================================================
# The training script periodically ran fixed prompts through the
# model and recorded its outputs. This shows how the model's
# responses evolved from random noise to structured classifications.
# ============================================================

import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

probe_path = f'{OUTPUT_DIR}/sft/probe_history.json'
if os.path.exists(probe_path):
    with open(probe_path) as f:
        probe_data = json.load(f)
    probes = probe_data.get('probes', [])

    if probes:
        # Group by label
        labels = sorted(set(p['expected'] for p in probes))
        steps_seen = sorted(set(p['step'] for p in probes))

        # --- Figure: Probe accuracy timeline ---
        fig, ax = plt.subplots(figsize=(14, 3))
        colors_map = {'Backup Archive': '#64748b', 'OLTP Database': '#f97316', 'VDI Virtual Desktop': '#a855f7'}

        for i, label in enumerate(labels):
            label_probes = [p for p in probes if p['expected'] == label]
            xs = [p['step'] for p in label_probes]
            ys = [i] * len(label_probes)
            cs = [colors_map.get(label, '#3b82f6') if p['correct'] else '#ef4444' for p in label_probes]
            markers = ['o' if p['correct'] else 'x' for p in label_probes]
            for x, y, c, m in zip(xs, ys, cs, markers):
                ax.scatter(x, y, c=c, marker=m, s=80, zorder=3)

        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=10)
        ax.set_xlabel('Training Step')
        ax.set_title('SFT Live Probe Timeline — When Did the Model Start Getting It Right?', fontsize=11)
        ax.grid(True, alpha=0.2, axis='x')
        ax.legend(handles=[
            mpatches.Patch(color='#22c55e', label='Correct (●)'),
            mpatches.Patch(color='#ef4444', label='Wrong (✗)'),
        ], loc='lower right', fontsize=9)
        plt.tight_layout()
        plt.show()

        # --- Text: Show model outputs at 3 key snapshots ---
        n = len(steps_seen)
        snapshot_steps = [steps_seen[0], steps_seen[n//2], steps_seen[-1]]
        print("Model Output Snapshots During SFT Training:")
        print("=" * 70)
        for snap_step in snapshot_steps:
            print(f"\n  Step {snap_step}:")
            step_probes = [p for p in probes if p['step'] == snap_step]
            for p in step_probes:
                badge = "✓" if p['correct'] else "✗"
                text = p['generated'][:90].replace('\n', ' | ')
                print(f"    {badge} {p['expected']:<22s} → {text}")
        print()
else:
    print("No probe history found — run SFT training first.")

---
## Step 2: Direct Preference Optimization (DPO)

**What's the problem?** The SFT model knows the task, but its outputs might be verbose, hedging, or inconsistent in style. DPO fixes this — think of it as **code review for model outputs**. It teaches the difference between a 3-page incident report and a crisp 2-line summary.

**Analogy:** You wouldn't retrain a new hire to do their job differently — you'd show them examples of good vs. bad reports and say "write like this, not like that." That's DPO.

**What happens:**
- Generates 300 preference pairs (chosen = correct label, rejected = wrong label)
- Trains a new LoRA adapter to increase the likelihood gap between chosen and rejected responses

**What to expect:**
- DPO typically does NOT improve accuracy — it may even drop a few points. That's OK.
- DPO optimizes for *style* (concise, confident), not *correctness*
- The key metric is whether the model's outputs become more concise and confident

**What to look for in the results:**
- Did the training loss decrease? (Means the model learned the preference signal)
- Is accuracy similar to SFT? (Expected — DPO refines, it doesn't teach new skills)
- Look at the chosen/rejected log-prob shifts — chosen should go up, rejected down

**Prerequisite:** Step 1 (SFT) must complete first
**Expected runtime:** See the **Model Configuration** cell above

In [ ]:
# ============================================================
# Run DPO training
# ============================================================
# Requires the SFT adapter from Step 1.
# Trains the model to prefer concise, correct responses.
#
# Outputs saved to scripts/outputs/ (path varies by model size)
# ============================================================

!python scripts/train_dpo.py {MODEL_SIZE_FLAG}

In [ ]:
# ============================================================
# DPO Probe History: Watch Style Refinement (Option C)
# ============================================================
# DPO doesn't change WHAT the model knows — it changes HOW it says it.
# The probe history shows whether the model maintains accuracy while
# its output style shifts.
# ============================================================

import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

probe_path = f'{OUTPUT_DIR}/dpo/probe_history.json'
if os.path.exists(probe_path):
    with open(probe_path) as f:
        probe_data = json.load(f)
    probes = probe_data.get('probes', [])

    if probes:
        labels = sorted(set(p['expected'] for p in probes))
        steps_seen = sorted(set(p['step'] for p in probes))

        # --- Figure: Probe accuracy timeline ---
        fig, ax = plt.subplots(figsize=(14, 3))
        colors_map = {'Backup Archive': '#64748b', 'OLTP Database': '#f97316', 'VDI Virtual Desktop': '#a855f7'}

        for i, label in enumerate(labels):
            label_probes = [p for p in probes if p['expected'] == label]
            xs = [p['step'] for p in label_probes]
            ys = [i] * len(label_probes)
            cs = [colors_map.get(label, '#3b82f6') if p['correct'] else '#ef4444' for p in label_probes]
            markers = ['o' if p['correct'] else 'x' for p in label_probes]
            for x, y, c, m in zip(xs, ys, cs, markers):
                ax.scatter(x, y, c=c, marker=m, s=80, zorder=3)

        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=10)
        ax.set_xlabel('Training Step')
        ax.set_title('DPO Live Probe Timeline — Does Style Refinement Preserve Accuracy?', fontsize=11)
        ax.grid(True, alpha=0.2, axis='x')
        ax.legend(handles=[
            mpatches.Patch(color='#22c55e', label='Correct (●)'),
            mpatches.Patch(color='#ef4444', label='Wrong (✗)'),
        ], loc='lower right', fontsize=9)
        plt.tight_layout()
        plt.show()

        # --- Text: Show model outputs at start vs end ---
        n = len(steps_seen)
        snapshot_steps = [steps_seen[0], steps_seen[-1]]
        print("Model Output: Start vs End of DPO Training:")
        print("=" * 70)
        for snap_step in snapshot_steps:
            print(f"\n  Step {snap_step}:")
            step_probes = [p for p in probes if p['step'] == snap_step]
            for p in step_probes:
                badge = "✓" if p['correct'] else "✗"
                text = p['generated'][:90].replace('\n', ' | ')
                print(f"    {badge} {p['expected']:<22s} → {text}")
        print("\n  Look for: same correctness, but tighter/more confident phrasing.")
else:
    print("No DPO probe history found — run DPO training first.")

In [ ]:
# ============================================================
# Post-DPO Validation: Did style refinement work?
# ============================================================

import json
import os
import matplotlib.pyplot as plt
import numpy as np

# --- Load artifacts ---
dpo_dir = f'{OUTPUT_DIR}/dpo'
sft_dir = f'{OUTPUT_DIR}/sft'

curves = None
curves_path = os.path.join(dpo_dir, 'training_curves.json')
if os.path.exists(curves_path):
    with open(curves_path) as f:
        curves = json.load(f)

dpo_sanity = None
dpo_sanity_path = os.path.join(dpo_dir, 'dpo_sanity_check.json')
if os.path.exists(dpo_sanity_path):
    with open(dpo_sanity_path) as f:
        dpo_sanity = json.load(f)

sft_sanity = None
sft_sanity_path = os.path.join(sft_dir, 'sft_sanity_check.json')
if os.path.exists(sft_sanity_path):
    with open(sft_sanity_path) as f:
        sft_sanity = json.load(f)

prob_shifts = None
prob_path = os.path.join(dpo_dir, 'probability_shifts.json')
if os.path.exists(prob_path):
    with open(prob_path) as f:
        prob_shifts = json.load(f)

# --- Two-panel figure ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: DPO training loss + reward margin
if curves and curves.get('loss_curve'):
    logs = curves['loss_curve']
    steps = [l['step'] for l in logs]
    losses = [l.get('loss', 0) for l in logs]
    ax1.plot(steps, losses, 'b-', linewidth=2, label='Loss')

    # Plot reward margins if available
    margins = [l.get('rewards/margins', None) for l in logs]
    if any(m is not None for m in margins):
        ax1_twin = ax1.twinx()
        valid_steps = [s for s, m in zip(steps, margins) if m is not None]
        valid_margins = [m for m in margins if m is not None]
        ax1_twin.plot(valid_steps, valid_margins, 'r--', linewidth=1.5, label='Reward Margin')
        ax1_twin.set_ylabel('Reward Margin', color='red')
        ax1_twin.legend(loc='upper right')

    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Loss', color='blue')
    ax1.set_title('DPO Training Curves')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left')
else:
    ax1.text(0.5, 0.5, 'No training curve data', ha='center', va='center', transform=ax1.transAxes)

# Right: SFT vs DPO accuracy comparison
sft_acc = sft_sanity.get('sample_accuracy', 0) if sft_sanity else 0
dpo_acc = dpo_sanity.get('sample_accuracy', 0) if dpo_sanity else 0

bars = ax2.bar(['SFT', 'DPO'], [sft_acc, dpo_acc],
               color=['#66b3ff', '#ff9933'], alpha=0.8, width=0.5)
ax2.set_ylabel('Sample Accuracy')
ax2.set_title('SFT vs DPO Accuracy')
ax2.set_ylim(0, 1.1)
for bar, val in zip(bars, [sft_acc, dpo_acc]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.0%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# --- Preference pair example with log-prob shift ---
if prob_shifts and prob_shifts.get('examples'):
    ex = prob_shifts['examples'][0]
    print("Example Preference Pair — Log-Probability Shift:")
    print("-" * 60)
    print(f"  Chosen style:    {ex.get('chosen_style', 'concise')}")
    print(f"  Rejected style:  {ex.get('rejected_style', 'wrong/verbose')}")
    print(f"  Before DPO:  chosen={ex['before']['chosen_log_prob']:.3f}  "
          f"rejected={ex['before']['rejected_log_prob']:.3f}  "
          f"gap={ex['before']['chosen_log_prob'] - ex['before']['rejected_log_prob']:.3f}")
    print(f"  After DPO:   chosen={ex['after']['chosen_log_prob']:.3f}  "
          f"rejected={ex['after']['rejected_log_prob']:.3f}  "
          f"gap={ex['after']['chosen_log_prob'] - ex['after']['rejected_log_prob']:.3f}")
    gap_before = ex['before']['chosen_log_prob'] - ex['before']['rejected_log_prob']
    gap_after = ex['after']['chosen_log_prob'] - ex['after']['rejected_log_prob']
    if gap_after > gap_before:
        print(f"  -> DPO widened the gap by {gap_after - gap_before:.3f} (chosen preferred more)")
    print()

# --- Interpretation ---
print("=" * 60)
print("INTERPRETATION")
print("=" * 60)
print(f"  SFT accuracy:  {sft_acc:.0%}")
print(f"  DPO accuracy:  {dpo_acc:.0%}")
delta = dpo_acc - sft_acc
if abs(delta) < 0.1:
    print(f"  Accuracy is similar ({'+' if delta >= 0 else ''}{delta:.0%}) — expected for DPO.")
    print(f"  DPO refines style (concise, confident), not correctness.")
elif delta < 0:
    print(f"  Accuracy dropped {delta:.0%} — this can happen with DPO.")
    print(f"  DPO trades some accuracy for style consistency.")
else:
    print(f"  Accuracy improved {delta:+.0%} — bonus from DPO preference learning.")

if dpo_sanity:
    print(f"\n  Verdict: {dpo_sanity.get('verdict', 'UNKNOWN')}")
print("=" * 60)

---
## Step 3: Group Relative Policy Optimization (GRPO)

**What's the problem?** SFT taught the model *what* to output. DPO refined *how* it outputs. But neither directly optimized for *getting the right answer*. GRPO does — it's like **A/B testing at scale**: generate 8 candidate answers, score them, and learn from the best ones.

**Analogy:** Imagine testing 8 different storage configurations for each workload, measuring which ones perform best, then keeping the winning settings. That's GRPO — trial, measurement, and refinement.

**What happens:**
- For each prompt, generates K=8 completions using sampling
- Scores each with a binary reward: correct classification = 1.0, wrong = 0.0
- Computes group-relative advantages and updates the policy with REINFORCE gradients

**What to expect:**
- This is the longest training step
- Accuracy should climb upward from the SFT baseline over training
- If it plateaus, the model has hit its capacity limit — a valid finding, not a bug

**What to look for in the results:**
- Does the accuracy curve trend upward?
- What's the final accuracy vs. SFT baseline?
- Which categories does the model still struggle with?

**Prerequisite:** Step 1 (SFT) must complete first (DPO is not required)

> **Timeout warning:** If Colab disconnects during GRPO, your SFT and DPO adapters are already saved. Reconnect, re-run Setup, then run GRPO in a new session.

**Expected runtime:** See the **Model Configuration** cell above

In [ ]:
# ============================================================
# Run GRPO training
# ============================================================
# Requires the SFT adapter from Step 1.
# Uses online RL with a binary reward function.
#
# If TRL >= 0.12 is installed, uses TRL's GRPOTrainer.
# Otherwise falls back to a simplified manual implementation.
#
# Outputs saved to scripts/outputs/ (path varies by model size)
# ============================================================

!python scripts/train_grpo.py {MODEL_SIZE_FLAG}

In [ ]:
# ============================================================
# Post-GRPO Validation: Did RL improve accuracy?
# ============================================================

import json
import os
import matplotlib.pyplot as plt
import numpy as np

# --- Load artifacts ---
grpo_dir = f'{OUTPUT_DIR}/grpo'
sft_dir = f'{OUTPUT_DIR}/sft'
dpo_dir = f'{OUTPUT_DIR}/dpo'

grpo_curves = None
curves_path = os.path.join(grpo_dir, 'training_curves.json')
if os.path.exists(curves_path):
    with open(curves_path) as f:
        grpo_curves = json.load(f)

grpo_stats = None
stats_path = os.path.join(grpo_dir, 'group_statistics.json')
if os.path.exists(stats_path):
    with open(stats_path) as f:
        grpo_stats = json.load(f)

grpo_sanity = None
grpo_sanity_path = os.path.join(grpo_dir, 'grpo_sanity_check.json')
if os.path.exists(grpo_sanity_path):
    with open(grpo_sanity_path) as f:
        grpo_sanity = json.load(f)

sft_sanity = None
sft_sanity_path = os.path.join(sft_dir, 'sft_sanity_check.json')
if os.path.exists(sft_sanity_path):
    with open(sft_sanity_path) as f:
        sft_sanity = json.load(f)

dpo_sanity = None
dpo_sanity_path = os.path.join(dpo_dir, 'dpo_sanity_check.json')
if os.path.exists(dpo_sanity_path):
    with open(dpo_sanity_path) as f:
        dpo_sanity = json.load(f)

# --- Two-panel figure ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy curve over training steps (THE money chart)
if grpo_stats and grpo_stats.get('accuracy_curve'):
    acc_data = grpo_stats['accuracy_curve']
    steps = [d['step'] for d in acc_data]
    accs = [d['accuracy'] for d in acc_data]
    ax1.plot(steps, accs, 'g-', linewidth=2, alpha=0.7)
    # Smoothed trend line
    if len(accs) > 5:
        window = min(5, len(accs) // 3)
        smoothed = np.convolve(accs, np.ones(window)/window, mode='valid')
        smooth_steps = steps[window-1:]
        ax1.plot(smooth_steps, smoothed, 'g-', linewidth=3, label='Trend')
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('GRPO Accuracy Over Training')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.05)
    if sft_sanity:
        sft_acc = sft_sanity.get('sample_accuracy', 0)
        ax1.axhline(y=sft_acc, color='blue', linestyle='--', alpha=0.5, label=f'SFT baseline ({sft_acc:.0%})')
    ax1.legend()
else:
    ax1.text(0.5, 0.5, 'No accuracy curve data', ha='center', va='center', transform=ax1.transAxes)

# Right: Cumulative comparison — base vs SFT vs DPO vs GRPO
model_names = []
model_accs = []

model_names.append('Base')
model_accs.append(0.0)  # Base is effectively random

if sft_sanity:
    model_names.append('SFT')
    model_accs.append(sft_sanity.get('sample_accuracy', 0))

if dpo_sanity:
    model_names.append('DPO')
    model_accs.append(dpo_sanity.get('sample_accuracy', 0))

if grpo_sanity:
    model_names.append('GRPO')
    model_accs.append(grpo_sanity.get('sample_accuracy', 0))

colors = ['#ff9999', '#66b3ff', '#ff9933', '#66cc66'][:len(model_names)]
bars = ax2.bar(model_names, model_accs, color=colors, alpha=0.8, width=0.5)
ax2.set_ylabel('Sample Accuracy')
ax2.set_title('Accuracy Progression: All Stages')
ax2.set_ylim(0, 1.15)
for bar, val in zip(bars, model_accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.0%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# --- Best and worst outputs ---
if grpo_sanity:
    best = grpo_sanity.get('best_generation')
    worst = grpo_sanity.get('worst_generation')
    if best:
        print(f"Best output:  [{best['label']}] {best['text']}")
    if worst:
        print(f"Worst output: [{worst['label']}] {worst['text']}")
    print()

# --- Interpretation ---
print("=" * 60)
print("INTERPRETATION")
print("=" * 60)

sft_acc = sft_sanity.get('sample_accuracy', 0) if sft_sanity else 0
grpo_acc = grpo_sanity.get('sample_accuracy', 0) if grpo_sanity else 0

if grpo_sanity:
    init_acc = grpo_sanity.get('initial_accuracy')
    final_acc = grpo_sanity.get('final_accuracy')
    if init_acc is not None and final_acc is not None:
        print(f"  Training accuracy: {init_acc:.0%} -> {final_acc:.0%}")
        if final_acc > init_acc:
            print(f"  RL reward signal drove accuracy upward.")
        else:
            print(f"  Accuracy plateaued — model may have hit capacity limit.")

delta = grpo_acc - sft_acc
print(f"  SFT -> GRPO: {'+' if delta >= 0 else ''}{delta:.0%} points")
if delta > 0:
    print(f"  GRPO successfully improved on SFT through reward optimization.")
elif delta == 0:
    print(f"  GRPO matched SFT — the model's capacity may be the bottleneck.")
else:
    print(f"  GRPO below SFT — investigate reward function or training stability.")

# Per-category analysis
if grpo_sanity and grpo_sanity.get('sample_results'):
    correct_cats = [r['expected'] for r in grpo_sanity['sample_results'] if r['correct']]
    wrong_cats = [r['expected'] for r in grpo_sanity['sample_results'] if not r['correct']]
    if correct_cats:
        print(f"\n  Reliably classifies: {', '.join(correct_cats)}")
    if wrong_cats:
        print(f"  Still struggles with: {', '.join(wrong_cats)}")

if grpo_sanity:
    print(f"\n  Verdict: {grpo_sanity.get('verdict', 'UNKNOWN')}")
print("=" * 60)

In [ ]:
# ============================================================
# GRPO Probe History: Watch RL Optimize Accuracy (Option C)
# ============================================================
# GRPO is the most dramatic to watch — the model literally
# teaches itself to get the right answer through trial and error.
# ============================================================

import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

probe_path = f'{OUTPUT_DIR}/grpo/probe_history.json'
if os.path.exists(probe_path):
    with open(probe_path) as f:
        probe_data = json.load(f)
    probes = probe_data.get('probes', [])

    if probes:
        labels = sorted(set(p['expected'] for p in probes))
        steps_seen = sorted(set(p['step'] for p in probes))

        # --- Figure: Probe accuracy timeline + running accuracy ---
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), gridspec_kw={'height_ratios': [2, 1]})
        colors_map = {'Backup Archive': '#64748b', 'OLTP Database': '#f97316', 'VDI Virtual Desktop': '#a855f7'}

        # Top: Per-category timeline
        for i, label in enumerate(labels):
            label_probes = [p for p in probes if p['expected'] == label]
            xs = [p['step'] for p in label_probes]
            ys = [i] * len(label_probes)
            cs = [colors_map.get(label, '#3b82f6') if p['correct'] else '#ef4444' for p in label_probes]
            markers = ['o' if p['correct'] else 'x' for p in label_probes]
            for x, y, c, m in zip(xs, ys, cs, markers):
                ax1.scatter(x, y, c=c, marker=m, s=80, zorder=3)

        ax1.set_yticks(range(len(labels)))
        ax1.set_yticklabels(labels, fontsize=10)
        ax1.set_title('GRPO Live Probe Timeline — Watch the Model Learn Through RL', fontsize=11)
        ax1.grid(True, alpha=0.2, axis='x')
        ax1.legend(handles=[
            mpatches.Patch(color='#22c55e', label='Correct (●)'),
            mpatches.Patch(color='#ef4444', label='Wrong (✗)'),
        ], loc='lower right', fontsize=9)

        # Bottom: Running probe accuracy
        acc_per_step = []
        for s in steps_seen:
            step_probes = [p for p in probes if p['step'] == s]
            if step_probes:
                acc_per_step.append(sum(1 for p in step_probes if p['correct']) / len(step_probes))
            else:
                acc_per_step.append(0)
        ax2.plot(steps_seen, acc_per_step, 'g-o', linewidth=2, markersize=4)
        ax2.fill_between(steps_seen, acc_per_step, alpha=0.15, color='green')
        ax2.set_xlabel('Training Step')
        ax2.set_ylabel('Probe Accuracy')
        ax2.set_ylim(-0.05, 1.1)
        ax2.grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

        # --- Text: Show model outputs at 3 key snapshots ---
        n = len(steps_seen)
        snapshot_steps = [steps_seen[0], steps_seen[n//2], steps_seen[-1]]
        print("Model Output Snapshots During GRPO Training:")
        print("=" * 70)
        for snap_step in snapshot_steps:
            step_probes_snap = [p for p in probes if p['step'] == snap_step]
            correct_count = sum(1 for p in step_probes_snap if p['correct'])
            print(f"\n  Step {snap_step} ({correct_count}/{len(step_probes_snap)} correct):")
            for p in step_probes_snap:
                badge = "✓" if p['correct'] else "✗"
                text = p['generated'][:90].replace('\n', ' | ')
                print(f"    {badge} {p['expected']:<22s} → {text}")
        print()
else:
    print("No GRPO probe history found — run GRPO training first.")

---
## Step 4: Export Artifacts

**What's the problem?** We've trained three model variants, but how do they actually compare? This step is like **running FIO against every storage controller with the same workload** — standardized benchmarking so you can compare apples to apples.

**What happens:** Loads each model variant (base, SFT, DPO, GRPO) and runs greedy inference on 20 identical test prompts. Produces a single `precomputed_results.json` for the web app.

**What to look for:**
- A clear progression: base (random) < SFT (learned) ≤ DPO (refined) < GRPO (optimized)
- If DPO accuracy is slightly below SFT, that's normal — DPO optimizes style, not accuracy
- If GRPO is the highest, the RL reward signal worked
- Check which categories the best model gets right vs. wrong

**Expected runtime:** See the **Model Configuration** cell above

In [ ]:
# ============================================================
# Run export
# ============================================================
# Evaluates all available model variants on 20 test prompts
# and produces precomputed_results.json for the web app.
#
# The script will skip any missing adapters and export
# results for whatever models are available.
#
# Outputs:
#   app/public/data/precomputed_results.json  - Main results file
#   app/public/data/resource_summary.json     - Resource utilization
#   scripts/outputs/export/                   - Backup copies
# ============================================================

!python scripts/export_artifacts.py {MODEL_SIZE_FLAG}

---
## Step 5: Convert to ONNX for Browser Inference

This step converts two model variants to ONNX format and pushes them to HuggingFace Hub, enabling **client-side inference** directly in the browser via `@huggingface/transformers` (transformers.js).

**Two models are converted:**
1. **Base model** (untrained) — shows the model's behavior before any training
2. **GRPO model** (SFT + GRPO merged) — shows the fully trained result

**What happens:**
- Saves the unmodified base model in a clean directory
- Merges SFT adapter → merges GRPO adapter on top → saves the merged model
- Converts both to ONNX using `optimum`
- Pushes both ONNX models to HuggingFace Hub as separate repos

**Prerequisites:** Steps 1 (SFT) and 3 (GRPO) must be complete. You must be logged in to HuggingFace Hub.
**Expected runtime:** See the **Model Configuration** cell above
**Output:** Two HuggingFace Hub repos with ONNX models

In [ ]:
# ============================================================
# Convert models to ONNX and push to HuggingFace Hub
# ============================================================
# Requires SFT and GRPO adapters from Steps 1 and 3.
# Converts base and merged GRPO models to ONNX format,
# then pushes to HuggingFace Hub for browser-side inference.
#
# If you skipped the optional HF login cell earlier, run it now.
# ============================================================

# Install ONNX conversion dependencies
!pip install -q optimum[onnxruntime] onnxruntime huggingface_hub

# Verify HF login (if not logged in, run the optional login cell above)
from huggingface_hub import HfApi
try:
    user = HfApi().whoami()
    print(f"Logged in as: {user['name']}")
except Exception:
    print("Not logged in! Run the HuggingFace login cell (after GPU check) first.")
    raise RuntimeError("HuggingFace login required for ONNX push")

# Run the conversion script
!python scripts/convert_to_onnx.py {MODEL_SIZE_FLAG}

---
## Final Validation

This cell loads `precomputed_results.json` and gives you the complete picture:

1. **Accuracy bar chart** — all model variants side-by-side
2. **Inference speed comparison** — generation time per variant
3. **Side-by-side outputs** — see exactly what each model generates for the same prompt
4. **Per-category breakdown** — which workload types the best model gets right vs. wrong
5. **Narrative interpretation** — plain-English summary of the training progression

**What to look for:**
- A clear progression: base (random) < SFT (learned) ≤ DPO (refined) < GRPO (optimized)
- Which categories are reliably classified vs. confused
- Whether the accuracy numbers tell a coherent story across stages

In [ ]:
# ============================================================
# Final Validation: Full Pipeline Results
# ============================================================

import json
import os
import matplotlib.pyplot as plt
import numpy as np

results_path = 'app/public/data/precomputed_results.json'
if not os.path.exists(results_path):
    results_path = 'scripts/outputs/export/precomputed_results.json'

with open(results_path) as f:
    results = json.load(f)

variants = results['metadata']['model_variants']
categories = results['metadata']['categories']

# --- 1. Horizontal bar chart of accuracy by model variant ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

accs = [results['model_results'][v]['summary']['accuracy'] for v in variants]
colors = {'base': '#ff9999', 'sft': '#66b3ff', 'dpo': '#ff9933', 'grpo': '#66cc66'}
bar_colors = [colors.get(v, '#cccccc') for v in variants]

bars = ax1.barh(variants, accs, color=bar_colors, alpha=0.8, height=0.5)
ax1.set_xlabel('Accuracy')
ax1.set_title('Accuracy by Model Variant')
ax1.set_xlim(0, 1.1)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.0%}', va='center', fontweight='bold')

# --- 2. Generation time comparison ---
gen_times = [results['model_results'][v]['summary']['avg_generation_time_ms'] for v in variants]
bars2 = ax2.barh(variants, gen_times, color=bar_colors, alpha=0.6, height=0.5)
ax2.set_xlabel('Avg Generation Time (ms)')
ax2.set_title('Inference Speed by Model Variant')
for bar, val in zip(bars2, gen_times):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}ms', va='center')

plt.tight_layout()
plt.show()

# --- 3. Table: 3 prompts showing all model outputs side-by-side ---
print("Side-by-Side Comparison (3 test prompts):")
print("=" * 80)
test_prompts = results['test_prompts'][:3]
for i, test in enumerate(test_prompts):
    print(f"\nPrompt {i+1}: True label = {test['true_label']}")
    print(f"  Metrics: IOPS={test['metrics']['iops']:,}, "
          f"Throughput={test['metrics']['throughput_mb']:,} MB/s, "
          f"Block={test['metrics']['block_size_kb']} KB")
    for v in variants:
        output = results['model_results'][v]['outputs'][i]
        text = output['generated_text'][:80].replace('\n', ' | ')
        print(f"  {v:>6s}: {text}")
    print()

# --- 4. Per-category accuracy for best model ---
best_variant = max(variants, key=lambda v: results['model_results'][v]['summary']['accuracy'])
best_outputs = results['model_results'][best_variant]['outputs']

print(f"Per-Category Results ({best_variant.upper()} model):")
print("-" * 50)
cat_correct = {c: 0 for c in categories}
cat_total = {c: 0 for c in categories}

for i, test in enumerate(results['test_prompts']):
    cat = test['true_label']
    cat_total[cat] += 1
    generated = best_outputs[i]['generated_text'].lower()
    if cat.lower() in generated:
        cat_correct[cat] += 1

for cat in categories:
    total = cat_total[cat]
    correct = cat_correct[cat]
    pct = correct / total if total > 0 else 0
    bar = "█" * int(pct * 10) + "░" * (10 - int(pct * 10))
    print(f"  {cat:<22s} {bar} {correct}/{total} ({pct:.0%})")

# --- 5. Final narrative interpretation ---
print()
print("=" * 60)
print("FINAL INTERPRETATION")
print("=" * 60)

variant_accs = {v: results['model_results'][v]['summary']['accuracy'] for v in variants}

if 'base' in variant_accs and 'sft' in variant_accs:
    delta = variant_accs['sft'] - variant_accs['base']
    print(f"  base -> sft:  Model learned the task "
          f"({'+' if delta >= 0 else ''}{delta:.0%} points)")

if 'sft' in variant_accs and 'dpo' in variant_accs:
    delta = variant_accs['dpo'] - variant_accs['sft']
    if abs(delta) < 0.05:
        print(f"  sft -> dpo:   Style refined, accuracy similar (expected)")
    else:
        print(f"  sft -> dpo:   Accuracy {'improved' if delta > 0 else 'dropped'} "
              f"({'+' if delta >= 0 else ''}{delta:.0%} points)")

if 'sft' in variant_accs and 'grpo' in variant_accs:
    delta = variant_accs['grpo'] - variant_accs['sft']
    print(f"  sft -> grpo:  RL {'improved' if delta > 0 else 'changed'} accuracy "
          f"({'+' if delta >= 0 else ''}{delta:.0%} points)")

best_acc = variant_accs.get(best_variant, 0)
correct_cats = [c for c in categories if cat_correct.get(c, 0) == cat_total.get(c, 0) and cat_total.get(c, 0) > 0]
wrong_cats = [c for c in categories if cat_correct.get(c, 0) < cat_total.get(c, 0) and cat_total.get(c, 0) > 0]

if correct_cats:
    print(f"\n  Reliably classifies: {', '.join(correct_cats)}")
if wrong_cats:
    print(f"  Struggles with: {', '.join(wrong_cats)}")

model_size = results['metadata'].get('model_size', '360M')
if best_acc < 0.6:
    print(f"\n  The {model_size} model has limited capacity for 6-class classification.")
    print(f"  This is a valid finding — a larger model (1.7B+) would help.")
else:
    print(f"\n  The {model_size} model achieves {best_acc:.0%} accuracy after post-training.")

print()
print(f"Training artifacts included:")
for key in ['training_data', 'dpo_preference_examples', 'grpo_generation_logs',
            'lora_weight_visualization', 'sft_before_after',
            'dpo_probability_shifts', 'grpo_group_statistics']:
    status = 'yes' if key in results else 'MISSING'
    print(f"  {key}: {status}")

file_size_mb = os.path.getsize(results_path) / 1e6
print(f"\nResults file: {file_size_mb:.2f} MB")
print("=" * 60)

In [ ]:
# ============================================================
# AI Research Advisor
# ============================================================
# Analyzes your training results and suggests next experiments.
# Requires an Anthropic API key.

!pip install -q anthropic rich

import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # Uncomment and paste your key

# Dry run first to see what will be sent to Claude
!python scripts/ai_advisor.py --dry-run

# Uncomment to get the actual recommendation:
# !python scripts/ai_advisor.py

---
## Validation

Verify that the exported results contain all expected model variants and check the accuracy summary. You should see:
- **base**: Low accuracy (the untrained model doesn't know the task)
- **sft**: Moderate-to-high accuracy (learned the classification format)
- **dpo**: Similar accuracy to SFT but with more concise/confident outputs
- **grpo**: Highest accuracy (optimized directly for classification correctness)

In [ ]:
# ============================================================
# Validate exported results
# ============================================================

import json
import os

results_path = 'app/public/data/precomputed_results.json'

if not os.path.exists(results_path):
    # Try the export directory backup
    results_path = 'scripts/outputs/export/precomputed_results.json'

with open(results_path) as f:
    results = json.load(f)

print(f"Models evaluated: {results['metadata']['model_variants']}")
print(f"Test prompts: {results['metadata']['num_test_prompts']}")
print(f"Categories: {results['metadata']['categories']}")
print(f"Export time: {results['metadata']['export_timestamp']}")
print()

print("Accuracy summary:")
for variant in results['metadata']['model_variants']:
    s = results['model_results'][variant]['summary']
    print(f"  {variant:>6s}: {s['accuracy']:.1%} ({s['correct']}/{s['total']}) "
          f"  avg_time={s['avg_generation_time_ms']:.1f}ms")

print()
print("Training artifacts included:")
for key in ['training_data', 'dpo_preference_examples', 'grpo_generation_logs',
            'lora_weight_visualization', 'sft_before_after',
            'dpo_probability_shifts', 'grpo_group_statistics',
            'resource_utilization']:
    status = 'yes' if key in results else 'MISSING'
    print(f"  {key}: {status}")

print()
file_size_mb = os.path.getsize(results_path) / 1e6
print(f"Results file size: {file_size_mb:.2f} MB")

In [ ]:
# ============================================================
# Download precomputed_results.json to your local machine
# ============================================================

from google.colab import files

download_path = 'app/public/data/precomputed_results.json'
if not os.path.exists(download_path):
    download_path = 'scripts/outputs/export/precomputed_results.json'

files.download(download_path)
print(f"Downloaded: {download_path}")

---
## Next Steps

1. **Copy the results file** to your local project:
   ```
   app/public/data/precomputed_results.json
   ```

2. **Update model IDs** in `app/src/services/inference.js` with the repo names printed by the ONNX conversion script:
   ```javascript
   const MODEL_CONFIGS = {
     base: { id: '<your-username>/smollm2-360m-storage-io-base-onnx', ... },
     grpo: { id: '<your-username>/smollm2-360m-storage-io-grpo-onnx', ... },
   }
   ```

3. **Update Epilogue links** in `app/src/stops/Epilogue.jsx` with your real HuggingFace/Colab URLs.

4. **Build the web app** to verify it picks up the real data:
   ```bash
   cd app
   npm install
   npm run build
   ```

5. **Run locally** to preview:
   ```bash
   npm run dev
   ```

The web app will automatically detect and use `precomputed_results.json` — no code changes needed for the tour and explore views. The LiveInferencePanel will download ONNX models from HuggingFace Hub for client-side browser inference.